# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dwaynemongaya/flyrank-ml-internship_dwaynemongaya/blob/main/work/notebooks/capstone.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

### Research Problem

Content managers often handle large inventories of webpages and may find it difficult to determine which pages should be reviewed first. Some pages may show declining search performance, low click-through rates, or outdated content, but manually reviewing every page requires time and effort.

### Research Question

Can observable content and search performance signals, combined with machine learning, help prioritize webpages for content refresh review?

### Decision Supported

This project aims to help content managers identify and prioritize webpages that may deserve further investigation.

The system uses a rule-based baseline and a Decision Tree model to rank potential content opportunities. Its output is intended to support human reviewers when deciding whether to refresh, improve, protect, or monitor a webpage.

The project does not automatically determine which pages must be changed. Final content decisions remain under human review.

### Research Objective

To evaluate whether a Decision Tree can provide useful ranking support compared with a simple rule-based baseline, using observable content and search performance signals from an anonymized starter dataset.

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

### Dataset Description

This project uses the anonymized `content_refresh_anonymized.csv` starter dataset provided by the FlyRank ML Internship.

The dataset contains approximately 30,000 content records and includes search performance, content metadata, engagement measurements, and trend information.

The dataset is designed to support research and educational use. Client and content identifiers are anonymized.

### Data Fields Used

The experiment uses the following six features:

| Feature | Description |
|---|---|
| content_age_days | Age of the content in days |
| days_since_last_update | Number of days since the last update |
| impressions_90d | Search impressions during the 90-day window |
| avg_position | Average search position |
| ctr | Click-through rate |
| word_count | Number of words in the content |

### Target Variable

The target is derived from `trend_direction`.

- `1` represents a record with a declining trend (`down`).
- `0` represents a record with another trend direction.

This is a proxy label based on the available trend measurements, not a separately observed future outcome.

### Data Exclusions

The following fields were excluded from the model features:

- `trend_direction`, because it is used to construct the target.
- `trend_pct`, because it represents the same trend outcome.
- `client_id`, because it is used for grouped validation rather than prediction.
- `content_id`, because it is an identifier rather than a meaningful performance signal.

### Warehouse Context

A separate Week 3 exercise examined the FlyRank pseudonymized warehouse release. Its daily performance table contains 78,835,655 rows covering January 27, 2025, through June 30, 2026.

However, the Decision Tree experiment in this paper uses the smaller starter dataset, not the full warehouse table.

### Data Privacy

No client names, original website addresses, private search queries, or identifying content information are included in the public research output.

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

### Research Approach

This study uses supervised machine learning to examine whether observable content and search performance signals can help prioritize webpages associated with a declining trend.

### Feature Selection

Six numerical features were selected:

1. `content_age_days`
2. `days_since_last_update`
3. `impressions_90d`
4. `avg_position`
5. `ctr`
6. `word_count`

These features represent content characteristics and observed search performance.

### Target Definition

The target variable is constructed as:

`trend_direction == "down"`

This means the model learns to classify records associated with the declining-trend category in the starter dataset.

It should be interpreted as a proxy classification task rather than a prediction of a future decline.

### Baseline Method

A simple rule-based baseline was established in Week 4.

A page receives 50 points when its search impressions are at least 500 and another 50 points when its average search position is 10 or better.

The resulting score ranges from 0 to 100.

### Machine Learning Method

A Decision Tree Classifier was selected because it can learn combinations of feature thresholds and provides an interpretable structure.

The model uses a maximum depth of four and balanced class weights to account for differences in class frequency.

### Validation Design

A grouped-by-client train/test split was used.

The dataset was divided so that observations from the same client did not appear in both training and testing.

The experiment used an 80% training and 20% testing allocation, with a fixed random seed for reproducibility.

### Evaluation Metric

Precision@20, Precision@50, and Precision@100 were used to evaluate the ranked results.

These metrics measure the proportion of declining-trend proxy records among the highest-ranked recommendations.

### Leakage and Assumptions

The direct target field and its trend percentage representation were excluded from the model features.

However, the selected search performance features and the target are derived from the same general reporting period. This creates a possible time-window overlap and limits the interpretation of the results.

A future experiment should use earlier performance measurements as features and a separate future period as the target.

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

In [ ]:

import pandas as pd
import numpy as np
import os
import json

from sklearn.model_selection import GroupShuffleSplit
from sklearn.tree import DecisionTreeClassifier

# Load the starter dataset
df = pd.read_csv("/content/content_refresh_anonymized.csv")

# Features and target from Week 5
features = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count"
]

X = (
    df[features]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

y = (df["trend_direction"] == "down").astype(int)
groups = df["client_id"]

# Grouped-by-client split
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

# Train Decision Tree
tree = DecisionTreeClassifier(
    max_depth=4,
    class_weight="balanced",
    random_state=42
)

tree.fit(X_train, y_train)

tree_scores = tree.predict_proba(X_test)[:, 1]

# Week 4 baseline
baseline_scores = (
    (X_test["impressions_90d"] >= 500).astype(int) * 50
    +
    (X_test["avg_position"] <= 10).astype(int) * 50
)

# Precision at K
def precision_at_k(scores, labels, k):
    scores = np.asarray(scores)
    labels = np.asarray(labels)

    order = np.argsort(-scores)
    return labels[order[:k]].mean()

# Compare baseline and Decision Tree
results = []

for k in [20, 50, 100]:
    results.append({
        "Metric": f"Precision@{k}",
        "Baseline": precision_at_k(
            baseline_scores, y_test, k
        ),
        "Decision Tree": precision_at_k(
            tree_scores, y_test, k
        )
    })

results_df = pd.DataFrame(results)

print("MODEL VS BASELINE RESULTS")
display(results_df.round(3))

# Validation information
train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Training clients:", len(train_clients))
print("Testing clients:", len(test_clients))
print("Client overlap:", len(train_clients & test_clients))


FileNotFoundError: [Errno 2] No such file or directory: '/content/content_refresh_anonymized.csv'

## 5. Limitations

*What this work cannot claim.*



This study has several limitations that should be considered when interpreting its results.

### Dataset Limitations

The experiment uses an anonymized starter dataset of approximately 30,000 records. It does not represent the entire FlyRank warehouse or every website and content inventory.

### Target Limitations

The target is derived from the existing `trend_direction` field. It represents a current-period declining-trend proxy rather than an independently measured future outcome.

### Possible Temporal Leakage

Some model features, including impressions, average position, and CTR, may overlap with the reporting period used to calculate the target.

Although the direct target field was excluded from the model inputs, this does not eliminate all possible leakage risks.

### Model Limitations

The Decision Tree can make incorrect classifications. It may miss records associated with declining trends or incorrectly identify records that are not declining.

Its learned patterns may also differ when evaluated on other clients or datasets.

### Generalization Limitations

Grouped-by-client validation provides a more careful assessment of performance on unseen clients. However, it does not establish that the model will perform equally well on all websites or future reporting periods.

### Practical Limitations

A high model score does not guarantee that a page needs a content refresh or that making changes will improve its search performance.

The system should therefore remain a decision-support prototype that requires human review.

### Overall Limitation

The findings are observational and directional. They demonstrate patterns in the available starter dataset but do not establish causation, future search performance, or guaranteed content improvement.

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

### Overview

The proposed recommendations use the ranked action playbook developed in Week 7.

The purpose is to help content managers identify which pages deserve attention first based on available content and search performance signals.

### Recommended Actions

| Priority | Reason Code | Recommended Action |
|---|---|---|
| High | DECLINE_SIGNAL | Review the page for signs of declining performance. |
| High | LOW_CTR_VISIBLE | Review the search snippet and content alignment. |
| Medium | STALE_VISIBLE | Check whether the page requires a content freshness review. |
| Monitoring | MONITOR | Continue observing performance before making changes. |

### Content Archetype-to-Action Mapping

| Content Archetype | Recommended Action |
|---|---|
| Potential Declining Pages | Review performance and investigate possible causes. |
| Low-CTR Visible Pages | Review title, description, and search intent alignment. |
| Stale Visible Pages | Evaluate whether content updates are appropriate. |
| High-Visibility Pages | Protect existing performance and monitor changes. |
| Low-Visibility Pages | Investigate search demand and content relevance before investing in updates. |

These archetypes are rule-based descriptions of observable patterns, not outputs from a separate clustering model.

### Recommended Workflow

1. Generate a ranked list of potential content opportunities.
2. Review the reason code behind each recommendation.
3. Examine search performance and content freshness.
4. Check for seasonality, changes in search demand, and related-page consolidation.
5. Select an appropriate action based on human judgment.
6. Monitor performance after any approved content changes.

### Expected Value

The playbook may reduce the time spent manually searching through large content inventories by organizing potential opportunities into a prioritized review queue.

However, the value of a recommendation depends on the reviewer's assessment, the effort required to make changes, and whether the identified issue represents a genuine opportunity.

### Human Oversight

No recommendation should automatically trigger content rewriting, deletion, publication, or website changes.

All actions require human approval.

The recommendations are intended to support content-management decisions, not replace professional judgment.

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

In [ ]:

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Create output folders
os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

# --------------------------------
# 1. Create the ranked action queue
# --------------------------------

queue = df.iloc[test_idx][["content_id"]].copy()

queue["model_score"] = tree_scores
queue["baseline_score"] = baseline_scores.values

# Assign reason codes
conditions = [
    queue["model_score"].values >= 0.50,
    (
        (X_test["impressions_90d"].values >= 500) &
        (X_test["ctr"].values < 0.5)
    ),
    (
        (X_test["days_since_last_update"].values >= 180) &
        (X_test["impressions_90d"].values >= 100)
    )
]

choices = [
    "DECLINE_SIGNAL",
    "LOW_CTR_VISIBLE",
    "STALE_VISIBLE"
]

queue["reason_code"] = np.select(
    conditions,
    choices,
    default="MONITOR"
)

# Map reasons to suggested actions
action_mapping = {
    "DECLINE_SIGNAL": "REVIEW_FOR_DECLINE",
    "LOW_CTR_VISIBLE": "REVIEW_SNIPPET",
    "STALE_VISIBLE": "REVIEW_FRESHNESS",
    "MONITOR": "MONITOR_PERFORMANCE"
}

queue["action"] = queue["reason_code"].map(action_mapping)

# Add archetype descriptions
queue["archetype"] = np.select(
    [
        queue["reason_code"] == "DECLINE_SIGNAL",
        queue["reason_code"] == "LOW_CTR_VISIBLE",
        queue["reason_code"] == "STALE_VISIBLE"
    ],
    [
        "Potential Declining Pages",
        "Low-CTR Visible Pages",
        "Stale Visible Pages"
    ],
    default="Monitoring Candidates"
)

queue["review_required"] = True

# Rank recommendations
ranked_queue = (
    queue.sort_values(
        ["model_score", "baseline_score"],
        ascending=False
    )
    .reset_index(drop=True)
)

ranked_queue.insert(
    0,
    "rank",
    range(1, len(ranked_queue) + 1)
)

# Export CSV
ranked_queue.to_csv(
    "work/outputs/w07_ranked_action_queue.csv",
    index=False
)

print("Ranked queue exported.")
display(ranked_queue.head(20))


# --------------------------------
# 2. Create model comparison chart
# --------------------------------

chart_data = results_df.set_index("Metric")

ax = chart_data.plot(
    kind="bar",
    figsize=(9, 5)
)

ax.set_title("Decision Tree vs Rule-Based Baseline")
ax.set_xlabel("Evaluation Metric")
ax.set_ylabel("Precision")
ax.set_ylim(0, 1)
ax.legend(title="Method")
plt.xticks(rotation=0)
plt.tight_layout()

plt.savefig(
    "work/figures/model_vs_baseline.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()


# --------------------------------
# 3. Create feature importance chart
# --------------------------------

importance_df = pd.DataFrame({
    "Feature": features,
    "Importance": tree.feature_importances_
}).sort_values("Importance", ascending=True)

plt.figure(figsize=(9, 5))

plt.barh(
    importance_df["Feature"],
    importance_df["Importance"]
)

plt.title("Decision Tree Feature Importance")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()

plt.savefig(
    "work/figures/feature_importance.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()


# --------------------------------
# 4. Save experiment metrics
# --------------------------------

metrics = {
    "dataset": "content_refresh_anonymized.csv",
    "total_rows": int(len(df)),
    "training_rows": int(len(train_idx)),
    "test_rows": int(len(test_idx)),
    "training_clients": int(groups.iloc[train_idx].nunique()),
    "test_clients": int(groups.iloc[test_idx].nunique()),
    "client_overlap": int(
        len(
            set(groups.iloc[train_idx]) &
            set(groups.iloc[test_idx])
        )
    ),
    "model": "Decision Tree",
    "target": "trend_direction == down",
    "features": features,
    "validation": "Grouped by client",
    "results": results_df.to_dict(orient="records"),
    "interpretation": "Decision-support prototype"
}

with open(
    "work/outputs/capstone_metrics.json",
    "w"
) as f:
    json.dump(metrics, f, indent=4)

print("\nAll capstone artifacts saved successfully.")

print("work/outputs/w07_ranked_action_queue.csv")
print("work/outputs/capstone_metrics.json")
print("work/figures/model_vs_baseline.png")
print("work/figures/feature_importance.png")


NameError: name 'df' is not defined

# 8. Five-Minute Project Demo Outline

**Project:** Evaluating Machine Learning for Content Refresh Opportunity Scoring

**Duration:** 5 minutes

### 1. Research Question (45 seconds)

Introduce the problem: content managers need an efficient way to identify which webpages deserve attention.

Research question:

Can observable content and search performance signals, combined with machine learning, help prioritize webpages for content refresh review?

### 2. Methodology (1 minute)

Explain the project workflow:

- Used an anonymized content performance starter dataset.
- Selected six numerical features related to content age, freshness, search impressions, average position, CTR, and word count.
- Defined the target using the existing declining-trend category.
- Developed a rule-based baseline.
- Trained a Decision Tree classifier.
- Used grouped-by-client validation to evaluate performance on unseen clients.

### 3. Visual Results (1 minute)

Present the Decision Tree feature importance chart.

Highlight that `impressions_90d` had the highest feature importance at approximately 0.535 in the Week-5 model.

Explain that feature importance describes the model's use of a variable, not a causal relationship.

### 4. One Honest Result (1 minute)

The grouped test contained 6,163 records.

The Decision Tree made 2,820 incorrect classifications, corresponding to an error rate of approximately 45.8%.

There were more false negatives than false positives, showing that the model missed a considerable number of records associated with the declining-trend proxy.

This demonstrates why model evaluation and human review are important.

### 5. Recommendations and Conclusion (1 minute 15 seconds)

Present the action playbook:

- Review potential declining pages.
- Investigate visible pages with low CTR.
- Check stale content that continues receiving impressions.
- Monitor pages where there is insufficient evidence for immediate changes.

Emphasize that recommendations require human review and should not automatically trigger content changes.

**Closing statement:**

This project demonstrates how machine learning can support content prioritization using observable search and content signals. Although the results show useful patterns, the model has limitations that must be considered before applying its recommendations. The final goal is to support better-informed content decisions, not replace human judgment.